# 7F Train, Stop, Restore, Continue

**Purpose:** Train a small image classifier, save a complete training checkpoint in a persistent workspace, stop the Klique VS Code session, then restore the files and continue in a new process.

**What we're doing**

Use scikit-learn's bundled handwritten digits: 1,797 grayscale **8 × 8** images and ten labels. A tiny PyTorch multilayer perceptron learns the task on CPU. No dataset download, model endpoint, credentials, or external model weights are needed. Package installation still needs approved network access or an internal mirror.

**Why we're doing it**

Workspace persistence saves files, not running Python processes. Model weights alone are not enough to continue the same training trajectory. This lab makes the boundary visible by preserving optimizer state, completed epochs, metrics, random state, and dataset/configuration identity.

**Output:** An epoch-3 checkpoint, a stopped session, an epoch-8 checkpoint produced after restoration, and held-out test metrics. GPU use is optional and is not required to complete the lab.

This is a guided UI lab. Run fenced commands in the **remote VS Code terminal from the repository root**, not in Python notebook cells. The included local tests validate training/checkpoint logic, not the Klique control plane; this notebook does **not** claim a live cloud stop/restore was tested.

## 7F.1 Prerequisites and Workspace Boundaries

* Complete [Running VS Code](07_Running_VS_Code.ipynb). Use an approved queue/image with Python **3.10+**, Git, and venv support. Package wheels must support the actual Python version and CPU architecture.
* CPU is the default. A small CPU development allocation is sufficient for the dataset; PyTorch packages can consume several GB, so check free storage before installation.
* Keep this checkout beneath the configured **Persistent Workspace Path**, such as `~/workspace`. If the launch form cloned it, enable **Store git repo**; the `git_repo_not_synced` symlink is not a repository backup.
* Record the original session ID, workspace path, queue, container image, Python version, and checkout commit. Choose an idle timeout that covers setup and training.
* Do not stop a session while a checkpoint is being written. Complete the requested training segment, verify the saved checkpoint, then stop deliberately.

```bash
pwd
hostname
python3 --version
uname -m
df -h .
git rev-parse HEAD
git status --short
```

Confirm `pwd` is the repository root within the configured persistent path. The default run directory is `07_VSCode/lab_outputs/digits-run`; it is ignored by Git. A local checkpoint or Git commit is not an independent backup.

See [Remote Development Environment](https://klique.ai/docs/latest/docs/orchestration_engine/gpuaas_remote_ides), [Launching a Remote IDE](https://klique.ai/docs/latest/docs/getting_started/launching_remote_ide/), and the [VS Code app reference](https://klique.ai/docs/latest/docs/webapp/applications/apps_vscode).

## 7F.2 Create the CPU Environment

Create a dedicated environment in the persistent workspace. Substitute your configured root for `~/workspace`. Do not reuse an unrelated environment.

```bash
python3 -m venv ~/workspace/.venvs/digits-lab
source ~/workspace/.venvs/digits-lab/bin/activate
python -m pip install 'torch>=2.6,<3' --index-url https://download.pytorch.org/whl/cpu
python -m pip install 'scikit-learn>=1.5,<2'
python -m pip check
python -c "import torch, sklearn; print('torch', torch.__version__, 'sklearn', sklearn.__version__); print('CPU lab selected')"
```

This CPU-index command is for platforms with a compatible official wheel. If no wheel is available, consult the [PyTorch installation selector](https://pytorch.org/get-started/locally/) and your administrator for the approved Python/architecture combination. Do not install host GPU drivers or change system permissions to complete this lab.

If venv support is absent, use an approved image with it or ask the administrator. Installing into the system Python is not the fallback.

> **Checkpoint:** `pip check` passes and the active interpreter imports both packages. Use this same interpreter for every command below.

## 7F.3 Understand the Training Contract

Read [`train_classifier.py`](train_classifier.py) before running it.

* It normalizes pixel values by the fixed constant 16 and creates deterministic, stratified **70% train / 15% validation / 15% test** partitions using seed 42. No statistics are fitted on the held-out data.
* The network is `64 → 64 → ReLU → Dropout(0.1) → 10`, with cross-entropy and Adam. Defaults: batch size 64 and learning rate 0.001. Validation reports progress; the test split is evaluated only when explicitly requested.
* Each completed epoch atomically replaces `checkpoint.pt` with the model state dict, full Adam state, completed epoch, history, Python/PyTorch random state, optional GPU random state, split/data hash, configuration, and software/device versions. Model architecture is defined in source, not pickled as an executable object.
* Resume reads using `torch.load(..., map_location='cpu', weights_only=True)`, validates schema and identity, loads model and optimizer state on the selected device, and restores random state. Never switch to `weights_only=False` or allowlist unknown checkpoint objects. Safe loading reduces code-execution exposure; it does not make arbitrary files trusted or prevent resource exhaustion. Load only your own checkpoints.
* `--epochs` is the **total target epoch**, not a count of additional epochs. Resume must advance beyond the checkpoint's completed epoch.
* Fresh runs require a **new directory**. There is no destructive overwrite flag. Resume deliberately updates an existing run only after validation. An exclusive lock prevents two training writers from sharing that run.
* `metrics.json` is a readable companion; `checkpoint.pt` is authoritative if interruption happens between the two atomic writes. A partial epoch is replayed from the last completed checkpoint, not continued mid-batch.

The implementation uses one CPU thread and deterministic PyTorch algorithms. Exact CPU resume equivalence is tested only under fixed software, hardware, and execution conditions. Different CPU/GPU types, drivers, platforms, or package builds can change floating-point results. The script rejects changed recorded package versions, device identity, training configuration, and dataset/split identity instead of silently pretending equivalence.

## 7F.4 Train the First Segment

From the repository root with the environment active:

```bash
python 07_VSCode/train_classifier.py --epochs 3
python -m pip freeze > 07_VSCode/lab_outputs/digits-run/environment.txt
python --version > 07_VSCode/lab_outputs/digits-run/python-version.txt
git rev-parse HEAD > 07_VSCode/lab_outputs/digits-run/source-commit.txt
python -m json.tool 07_VSCode/lab_outputs/digits-run/metrics.json
```

Expected: one JSON metrics line per epoch, then `Saved .../checkpoint.pt at epoch 3`. The process exits back to the shell. Accuracy should generally rise with training, but a particular score is not a restore criterion or guarantee.

Verify the authoritative file safely:

```bash
python - <<'PY'
from pathlib import Path
import torch
p = Path('07_VSCode/lab_outputs/digits-run/checkpoint.pt')
c = torch.load(p, map_location='cpu', weights_only=True)
assert c['last_epoch'] == 3
assert len(c['history']) == 3
print('Checkpoint:', p.resolve(), 'bytes:', p.stat().st_size)
print('Completed epoch:', c['last_epoch'], 'next epoch:', c['last_epoch'] + 1)
print('Dataset:', c['dataset'])
print('Versions:', c['versions'])
PY
```

The package freeze is reconstruction information, not a portability guarantee. Record the approved image and Python version separately. If a run already exists, inspect it and resume it or choose `--run-dir 07_VSCode/lab_outputs/digits-run-2`; use that same path in every later command.

> **Checkpoint:** The saved epoch is 3, the files are under the persistent root, and the shell prompt has returned. Do not select Stop before these checks pass.

## 7F.5 Stop the Session, Not Just the Browser

1. Save editor buffers. Record the session ID and exact output path outside the session, alongside your image/queue notes.
2. In Klique, return to the instance list and choose **Stop** from the instance action menu. Wait for stopped status and any documented snapshot completion indication. Closing the browser tab alone does not stop compute.
3. Keep the stopped instance. **Delete** is not part of this exercise.
4. Select **Clone**, review the prefilled configuration, and launch a new session. Alternatively, launch a new VS Code instance with **Restore Interactive Workspace ID** set to the recorded ID.
5. Retain the approved image, workspace configuration and device type. Open the new IDE and locate the restored repository/output directory.

The [Application Instance Actions](https://klique.ai/docs/latest/docs/webapp/applications/apps_overview#application-instance-actions) and [Launching a Remote IDE](https://klique.ai/docs/latest/docs/getting_started/launching_remote_ide/) describe this lifecycle. Administrator customization and storage policy may change the visible fields; check the current form rather than guessing.

> **Checkpoint:** The old instance is stopped, the new instance has its own session ID, and `checkpoint.pt` plus `environment.txt` exist in the restored workspace. This is a new process environment, not a suspended training process.

## 7F.6 Rebuild the Environment and Continue

Open a new terminal at the restored repository root. Check `python3 --version` against `python-version.txt`. Use the same recorded image and Python version. Do not assume a restored virtual environment remains valid: absolute paths, native libraries, and interpreter links can change.

Create a **new** environment path for the restored session. This CPU example reconstructs pinned packages using the approved package sources and official CPU wheel index. Use your organization's approved mirror instead if required.

```bash
python3 -m venv ~/workspace/.venvs/digits-lab-restored
source ~/workspace/.venvs/digits-lab-restored/bin/activate
python -m pip install -r 07_VSCode/lab_outputs/digits-run/environment.txt --extra-index-url https://download.pytorch.org/whl/cpu
python -m pip check
python 07_VSCode/train_classifier.py --resume --epochs 8 --evaluate-test
```

Expected first line: **`Resuming after epoch 3; next epoch 4`**. Epochs 4–8 follow. `test_metrics.json` reports held-out accuracy at epoch 8; do not tune hyperparameters against this test score. If evaluating at another target later, the file is replaced and explicitly labels its epoch.

```bash
python -m json.tool 07_VSCode/lab_outputs/digits-run/test_metrics.json
python - <<'PY'
import torch
c = torch.load('07_VSCode/lab_outputs/digits-run/checkpoint.pt', map_location='cpu', weights_only=True)
assert c['last_epoch'] == 8
assert [r['epoch'] for r in c['history']] == list(range(1, 9))
print('Verified continuous history through epoch', c['last_epoch'])
PY
```

> **Checkpoint:** The next epoch was 4, the final checkpoint reports 8, and all eight history entries are present. This demonstrates application-level continuation from saved files. It does not mean the original process survived.

If the restored venv path already exists from a prior attempt, inspect it before reuse or choose another fresh path. Do not delete the checkpoint to resolve an environment mismatch.

## 7F.7 Run the Local Regression Tests

```bash
python -m unittest discover -s 07_VSCode -p test_training.py -v
```

The tests use temporary directories and CPU only. They compare a four-epoch uninterrupted run with a two-epoch run followed by resume to four, including exact tensors, optimizer state, random state, and history. They also check accidental overwrite refusal, stale configuration/version/dataset rejection, malformed checkpoints, non-advancing epoch targets, concurrent-writer locks, and interrupted atomic writes.

These are executable tests of the lab code, not evidence that your Klique storage backend restored a snapshot. Your UI stop/restore and file checks establish that separately.

> **Checkpoint:** All tests pass in the same recorded environment. If exact replay fails, investigate software/hardware differences rather than loosening assertions to hide the difference.

## 7F.8 Optional GPU Experiment

Do this only when you have approval to allocate a GPU and install the matching packages. For this tiny dataset, GPU overhead may exceed any compute benefit; no speedup is promised.

* Use an approved GPU queue/image. Match NVIDIA CUDA or AMD ROCm wheels to the host driver, architecture, Python, and device. ARM64/GB10 and x86-64 use different compatible builds. Consult the [PyTorch selector](https://pytorch.org/get-started/locally/) and, for AMD, [ROCm's PyTorch guidance](https://rocm.docs.amd.com/projects/ai-ecosystem/en/latest/frameworks/pytorch/install.html).
* Create a separate environment with the approved GPU PyTorch build in the `>=2.6,<3` range and scikit-learn `>=1.5,<2`. The earlier CPU-only installation is not a GPU setup recipe. Do not modify host drivers or device permissions.
* `torch.cuda` is PyTorch's device API for both supported CUDA and ROCm builds. Confirm actual visibility before training. Apple MPS is not implemented in this lab.

```bash
python -c "import torch; print('torch:', torch.__version__, 'CUDA:', torch.version.cuda, 'HIP:', torch.version.hip); print('available:', torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No supported GPU')"
export CUBLAS_WORKSPACE_CONFIG=:4096:8
python 07_VSCode/train_classifier.py --device cuda --run-dir 07_VSCode/lab_outputs/digits-gpu-run --epochs 3
```

The workspace variable is required by this lab's GPU path for deterministic NVIDIA operations; it does not itself make ROCm or cross-device replay equivalent. If deterministic algorithms are unsupported on the chosen backend, use the CPU lab instead of silently disabling checks.

For GPU stop/restore, repeat the earlier verification, package-freeze, and UI lifecycle steps using the GPU run directory and approved GPU image. Rebuild with the **matching GPU package source**, not the CPU wheel command, and set the variable before Python starts. Then:

```bash
export CUBLAS_WORKSPACE_CONFIG=:4096:8
python 07_VSCode/train_classifier.py --device cuda --run-dir 07_VSCode/lab_outputs/digits-gpu-run --resume --epochs 8 --evaluate-test
```

GPU checkpoints are safely loaded through CPU storage and then restored onto the selected GPU. Resume requires the same recorded device name/build; it intentionally does not accept switching a CPU run to GPU or a GPU run to CPU. Even matching metadata cannot guarantee identical results across driver changes. CPU/GPU scores need not match bit-for-bit, and this lab's automated equivalence test covers CPU only.

## Troubleshooting

| Symptom | Check and recovery |
|---|---|
| Fresh run says directory exists | Inspect its checkpoint. Use `--resume` with a larger total epoch target or deliberately choose a new run directory. No overwrite is needed. |
| Target epochs must exceed saved epoch | `--epochs` is total, so epoch 8 requires a target above 8 to train further. |
| Configuration, dataset, device or version mismatch | Restore original arguments, code, image and pinned packages. For a different experiment, start a new directory. Do not edit checkpoint metadata to defeat validation. |
| Cannot safely load / malformed schema | Check that this is your lab-generated file from a completed write. Recover a known-good backup or start a new run; never enable unsafe pickle loading. |
| Lock remains after interrupted session | Read `.training.lock` and check for an active trainer. Only after verifying none is active, remove that single stale lock file and resume. Never remove a lock to permit simultaneous writers. |
| Files missing after restore | Check original workspace ID, persistent path, Store git repo, snapshot completion, and administrator storage policy. A file outside the captured root cannot be restored by this lab. |
| Lost newest epoch after interruption | Resume uses the last complete atomic checkpoint. Work since that boundary is replayed; epochs without a completed checkpoint were not durable. |
| Metrics JSON lags the checkpoint | The checkpoint is authoritative. A successful subsequent epoch regenerates the metrics file. |
| Package build unavailable or Python mismatch | Use the recorded approved image/Python and mirror. A restored venv is not evidence of compatibility. |
| GPU unavailable or deterministic operation fails | Verify approved allocation/build; use CPU in a fresh run if GPU support is unsuitable. |
| Out of space / snapshot too large | Stop training safely, inspect free space and storage policy, and preserve checkpoint/package records before any selective cleanup. |

## 7F.9 Save Results and Release Compute

1. Save the final checkpoint, readable metrics, environment freeze, source commit, and original/restored session IDs in the configured persistent workspace. Keep an approved independent backup if durability matters.
2. Inspect `git status --short` and `git check-ignore 07_VSCode/lab_outputs/digits-run/checkpoint.pt`. Do not commit training binaries, environment directories, or secrets.
3. Confirm training has exited. Stop the restored VS Code instance in Klique and verify stopped status. Stop any separately launched GPU instance too.
4. No recursive deletion is required. Remove only generated files/environments you deliberately choose to discard under storage policy; do not delete the instance needed for future restoration.

**Back:** [VS Code introduction](07_Running_VS_Code.ipynb)  
**Index:** [Repository README](../README.md)
